# Credit-Card Fraud Baseline

Train a reproducible, class-balanced logistic regression pipeline on `creditcard.csv`. The pipeline owns scaling and feature ordering so the same transformations can be used by the FastAPI inference layer.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'creditcard.csv'
ARTIFACT_DIR = ROOT / 'backend' / 'ml' / 'artifacts' / 'creditcard_baseline'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

frame = pd.read_csv(DATA_PATH)
feature_columns = [column for column in frame.columns if column != 'Class']
X = frame[feature_columns]
y = frame['Class'].astype('int8')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        class_weight='balanced', max_iter=2000, random_state=42, n_jobs=-1
    )),
])
pipeline.fit(X_train, y_train)
probabilities = pipeline.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype('int8')

metrics = {
    'model_name': 'creditcard_logistic_regression',
    'model_version': '2026-07-23.baseline.1',
    'feature_columns': feature_columns,
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'positive_train_rows': int(y_train.sum()),
    'positive_test_rows': int(y_test.sum()),
    'precision': float(precision_score(y_test, predictions, zero_division=0)),
    'recall': float(recall_score(y_test, predictions, zero_division=0)),
    'pr_auc': float(average_precision_score(y_test, probabilities)),
    'roc_auc': float(roc_auc_score(y_test, probabilities)),
    'confusion_matrix': confusion_matrix(y_test, predictions).tolist(),
}

joblib.dump(pipeline, ARTIFACT_DIR / 'model.joblib')
(ARTIFACT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
metrics

{'model_name': 'creditcard_logistic_regression',
 'model_version': '2026-07-23.baseline.1',
 'feature_columns': ['Time',
  'V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6',
  'V7',
  'V8',
  'V9',
  'V10',
  'V11',
  'V12',
  'V13',
  'V14',
  'V15',
  'V16',
  'V17',
  'V18',
  'V19',
  'V20',
  'V21',
  'V22',
  'V23',
  'V24',
  'V25',
  'V26',
  'V27',
  'V28',
  'Amount'],
 'train_rows': 227845,
 'test_rows': 56962,
 'positive_train_rows': 394,
 'positive_test_rows': 98,
 'precision': 0.06097560975609756,
 'recall': 0.9183673469387755,
 'pr_auc': 0.7189705771419241,
 'roc_auc': 0.9720834996210077,
 'confusion_matrix': [[55478, 1386], [8, 90]]}